# Ch 24. Decoding Strategies — Solutions

> This notebook contains reproducible solutions to all five exercises.


## Problem 1

**Problem**: Compare generation from the same prompt using Greedy, Top-k=5, Top-p=0.9, and Temperature=0.7.

### Solution

Greedy deterministically selects the maximum. The others sample from a normalized candidate set. A fair comparison uses the same logits and seed and reports both unique-token count and log probability.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
rng=np.random.default_rng(24); z=np.array([3.,2.2,1.5,1.,.4,-.2]);
def sample(z, mode):
 p=np.exp(z-z.max()); p/=p.sum()
 if mode=='greedy': return int(p.argmax())
 if mode=='topk': idx=np.argsort(p)[-5:]
 else:
  idx=np.argsort(p)[::-1]; idx=idx[np.cumsum(p[idx])-p[idx] < .9]
 q=p[idx]/p[idx].sum(); return int(rng.choice(idx,p=q))
out={m:[sample(z/.7 if m=='temp' else z, 'topp' if m=='temp' else m) for _ in range(200)] for m in ['greedy','topk','topp','temp']}
assert len(set(out['greedy']))==1 and all(0<=x<6 for v in out.values() for x in v)
{k:len(set(v)) for k,v in out.items()}


## Problem 2

**Problem**: Compare entropy and generation diversity for temperatures 0.3, 0.7, 1.0, and 1.5.

### Solution

Temperature $T$ gives $p_i(T)=\mathrm{softmax}(z_i/T)$. Lower $T$ concentrates the distribution and generally reduces entropy and diversity; higher $T$ flattens it and increases them.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
import numpy as np
z=np.array([3.,2.,1.,0.,-1.]); rows=[]
for T in [.3,.7,1.,1.5]:
 p=np.exp(z/T-(z/T).max()); p/=p.sum(); H=-np.sum(p*np.log(p)); rows.append((T,H,1/np.sum(p*p)))
assert all(rows[i][1] < rows[i+1][1] for i in range(3)); rows


## Problem 3

**Problem**: Compare Beam Search with beam widths 1, 4, and 8.

### Solution

Beam search retains the top $B$ prefixes by cumulative log probability. $B=1$ is greedy; increasing $B$ cannot worsen the search score, but may cause length bias and lower diversity.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
trans={():[(0,-.1),(1,-.2)],(0,):[(0,-2),(1,-.1)],(1,):[(0,-.2),(1,-.4)]}
def beam(B):
 beams=[((),0.)]
 for _ in range(2):
  cand=[]
  for seq,s in beams:
   opts=trans.get(seq,[(0,-.3),(1,-.3)])
   cand += [(seq+(t,),s+lp) for t,lp in opts]
  beams=sorted(cand,key=lambda x:x[1],reverse=True)[:B]
 return beams[0]
scores=[beam(B)[1] for B in [1,4,8]]
assert scores[0] <= scores[1] <= scores[2]; scores


## Problem 4

**Problem**: Experiment with the effects of Repetition Penalty values 1.0, 1.1, and 1.3.

### Solution

Use common random numbers: every penalty reuses the same uniform stream for each seed. Report the repetition-rate mean and sample standard deviation over 64 seeds, plus the paired mean difference and standard error versus penalty 1.0. Do not require noisy sample means to be perfectly monotone; instead verify the exact one-step repeat probability independently and test the paired effect of the strongest penalty.

The code is a small download-free CPU experiment and prints statistics computed by the run.


In [ ]:
import numpy as np

base = np.array([2.5, 2.0, 1.0, 0.0])
penalties = (1.0, 1.1, 1.3)

def repeat_rate(penalty, uniforms, steps=400):
    seq = []
    for u in uniforms[:steps]:
        logits = base.copy()
        if seq:
            repeated = np.unique(seq)
            positive = logits[repeated] > 0
            logits[repeated[positive]] /= penalty
            logits[repeated[~positive]] *= penalty
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        seq.append(int(np.searchsorted(np.cumsum(probs), u, side='right')))
    return np.mean(np.asarray(seq[1:]) == np.asarray(seq[:-1]))

seed_rates = {p: [] for p in penalties}
for seed in range(64):
    matched_uniforms = np.random.default_rng(seed).random(400)
    for penalty in penalties:
        seed_rates[penalty].append(repeat_rate(penalty, matched_uniforms))

summary = {
    p: {'mean': float(np.mean(v)), 'std': float(np.std(v, ddof=1))}
    for p, v in seed_rates.items()
}
paired = np.asarray(seed_rates[1.3]) - np.asarray(seed_rates[1.0])
summary['paired_1.3_minus_1.0'] = {
    'mean': float(paired.mean()),
    'standard_error': float(paired.std(ddof=1) / np.sqrt(len(paired))),
}

def probability_of_immediate_repeat(penalty, previous=0):
    logits = base.copy()
    logits[previous] /= penalty
    probs = np.exp(logits - logits.max())
    return float(probs[previous] / probs.sum())

reference = [probability_of_immediate_repeat(p) for p in penalties]
assert reference[0] > reference[1] > reference[2]
assert summary['paired_1.3_minus_1.0']['mean'] < 0
summary


## Problem 5

**Problem**: Explain mathematically why Speculative Decoding is 2–3x faster.

### Solution

A draft model proposes $k$ tokens and the target verifies them in parallel. If the mean accepted fraction is $a$, target-call cost per token is roughly $1/(1+ak)$ and speedup including draft cost $c_d$ is $S\approx(1+ak)/(1+k c_d)$.

The code below is a small CPU experiment with a fixed random seed. It makes no claim about absolute timing or large-scale quality; it verifies the mathematical quantities and invariants that must be compared.


In [ ]:
def speedup(k,accept,draft_cost): return (1+k*accept)/(1+k*draft_cost)
vals=[speedup(4,a,.05) for a in [.5,.7,.9]]
assert vals[0] > 2 and vals[-1] < 4; vals
